In [ ]:
!pip install requests beautifulsoup4 lxml

In [ ]:
#!/usr/bin/env python3
"""
IIT People Directory Scraper
=============================
Scrapes https://www.iit.edu/directory/people and outputs a CSV with
the exact same columns as Contacts_data.csv:

  Name, Department, Category, Description, Phone, Fax, Email,
  Building, Address, City, State, Zip, Source_url

Mapped as:
  Name        → person's full name
  Department  → their department (from profile page)
  Category    → Faculty or Staff
  Description → their job title(s)
  Phone       → phone number
  Fax         → (blank — people don't have fax listings)
  Email       → email address
  Building    → office building name
  Address     → street address
  City        → city
  State       → state
  Zip         → zip code
  Source_url  → their IIT profile URL

Requirements:
    pip install requests beautifulsoup4 lxml tqdm

Usage (Jupyter / Google Colab): just run the cell.
Edit the config block below to tune settings.
"""

import csv
import json
import logging
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ── Configuration — edit these values directly ─────────────────────────────────
class args:
    workers    = 10     # concurrent threads (lower to 5 if you see errors)
    delay      = 0.1    # per-worker pause in seconds
    output     = "iit_people_directory.csv"
    checkpoint = "iit_checkpoint.json"
    resume     = False  # set True to skip already-fetched profiles
    timeout    = 15     # HTTP timeout in seconds
# ──────────────────────────────────────────────────────────────────────────────

logging.basicConfig(level=logging.WARNING,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

BASE_URL      = "https://www.iit.edu"
DIRECTORY_URL = f"{BASE_URL}/directory/people"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}
MAX_RETRIES = 4

# Exact same columns as Contacts_data.csv
COLUMNS = [
    "Name", "Department", "Category", "Description",
    "Phone", "Fax", "Email",
    "Building", "Address", "City", "State", "Zip",
    "Source_url",
]

# ── Thread-local HTTP sessions ─────────────────────────────────────────────────
_local = threading.local()

def get_session() -> requests.Session:
    if not hasattr(_local, "session"):
        s = requests.Session()
        s.headers.update(HEADERS)
        retry = requests.adapters.Retry(
            total=MAX_RETRIES,
            backoff_factor=0.4,
            status_forcelist={429, 500, 502, 503, 504},
            allowed_methods={"GET"},
        )
        s.mount("https://", requests.adapters.HTTPAdapter(max_retries=retry))
        s.mount("http://",  requests.adapters.HTTPAdapter(max_retries=retry))
        _local.session = s
    return _local.session


def fetch(url: str) -> BeautifulSoup | None:
    s = get_session()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = s.get(url, timeout=args.timeout)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return BeautifulSoup(r.text, "lxml")
        except requests.RequestException as exc:
            if attempt < MAX_RETRIES:
                time.sleep(0.5 * attempt)
            else:
                log.error("Gave up on %s: %s", url, exc)
    return None


# ── Helpers ────────────────────────────────────────────────────────────────────
def clean(text) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text).strip())


def split_address(raw: str) -> tuple[str, str, str, str]:
    """
    Split a raw address string into (street, city, state, zip).
    Handles: '10 West 35th Street, Suite 2C4-2, Chicago, IL 60616'
    """
    raw = clean(raw)
    m = re.search(
        r"^(.*?),?\s*([A-Za-z ]+),\s*([A-Z]{2})\s+(\d{5}(?:-\d{4})?)$", raw
    )
    if m:
        return clean(m.group(1)), clean(m.group(2)), clean(m.group(3)), clean(m.group(4))
    return raw, "", "", ""


# ── Listing page parser ────────────────────────────────────────────────────────
def parse_listing_page(soup: BeautifulSoup) -> list[dict]:
    """Pull basic fields from each person card on a listing page."""
    people = []
    cards = soup.select("div.views-row, article.views-row")
    if not cards:
        links = soup.select('h3 a[href*="/directory/people/"]')
        cards = [a.find_parent() or a for a in links]

    for card in cards:
        p = {col: "" for col in COLUMNS}

        name_tag = (card.select_one('h3 a[href*="/directory/people/"]') or
                    card.select_one('a[href*="/directory/people/"]'))
        if not name_tag:
            continue

        p["Name"]       = clean(name_tag.get_text())
        p["Source_url"] = urljoin(BASE_URL, name_tag["href"])

        # Category = first profile type tag (Faculty / Staff)
        tag = card.select_one('a[href*="profile_type"]')
        if tag:
            p["Category"] = clean(tag.get_text())

        # Description = all role/title lines joined with semicolon
        roles = [
            clean(li.get_text())
            for li in card.select("ul li")
            if not li.select_one('a[href*="profile_type"]') and clean(li.get_text())
        ]
        p["Description"] = "; ".join(roles)

        # Phone (first one on the card)
        ph = card.select_one('a[href^="tel:"]')
        if ph:
            p["Phone"] = clean(ph.get_text())

        # Email
        em = card.select_one('a[href^="mailto:"]')
        if em:
            p["Email"] = em["href"].replace("mailto:", "").strip()

        people.append(p)
    return people


# ── Profile page parser ────────────────────────────────────────────────────────
def parse_profile_page(soup: BeautifulSoup, person: dict) -> None:
    """Enrich person dict in-place with department, building, address from profile."""

    # Department
    dept_tag = soup.select_one(
        "div.field--name-field-department .field__item, "
        "div.field--name-field-organization .field__item"
    )
    if dept_tag:
        person["Department"] = clean(dept_tag.get_text())
    else:
        dept_links = soup.select("div.field--name-field-organization a")
        depts = [
            clean(a.get_text()) for a in dept_links
            if clean(a.get_text()) and "directory" not in a.get("href", "")
        ]
        if depts:
            person["Department"] = depts[0]

    # Building
    building_tag = soup.select_one(
        "div.field--name-field-building .field__item, "
        "div.field--name-field-office-building .field__item"
    )
    if building_tag:
        person["Building"] = clean(building_tag.get_text())

    # Address block
    address_tag = soup.select_one(
        "div.field--name-field-address, "
        "div.field--name-field-office-address, "
        "div.field--name-field-location"
    )
    if address_tag:
        lines = [l.strip() for l in address_tag.get_text().splitlines() if l.strip()]
        if lines:
            # If building still empty, first line is often the building name
            if not person["Building"] and len(lines) > 1:
                person["Building"] = lines[0]
                raw = ", ".join(lines[1:])
            else:
                raw = ", ".join(lines)
            street, city, state, zip_ = split_address(raw)
            if street:
                person["Address"] = street
            if city:
                person["City"]  = city
            if state:
                person["State"] = state
            if zip_:
                person["Zip"]   = zip_

    # Phone — upgrade if missing from listing card
    if not person["Phone"]:
        ph = soup.select_one('a[href^="tel:"]')
        if ph:
            person["Phone"] = clean(ph.get_text())

    # Email — upgrade if missing
    if not person["Email"]:
        em = soup.select_one('a[href^="mailto:"]')
        if em:
            person["Email"] = em["href"].replace("mailto:", "").strip()

    # Default city/state/zip for IIT main campus when address is present but city isn't
    if person["Address"] and not person["City"]:
        person["City"]  = "Chicago"
        person["State"] = "IL"
        person["Zip"]   = "60616"


# ── Checkpoint helpers ─────────────────────────────────────────────────────────
def load_checkpoint() -> dict:
    p = Path(args.checkpoint)
    if p.exists():
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_checkpoint(data: dict) -> None:
    with open(args.checkpoint, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)


# ── Thread workers ─────────────────────────────────────────────────────────────
def fetch_listing_page(page_num: int) -> list[dict]:
    soup = fetch(f"{DIRECTORY_URL}?page={page_num}")
    time.sleep(args.delay)
    return parse_listing_page(soup) if soup else []


def fetch_profile(person: dict) -> dict:
    soup = fetch(person["Source_url"])
    time.sleep(args.delay)
    if soup:
        parse_profile_page(soup, person)
    return person


# ── Main ───────────────────────────────────────────────────────────────────────
def main() -> None:
    t0 = time.time()

    # Step 0 — discover total pages
    print("🔍  Fetching page 1 to discover total pages …")
    first_soup = fetch(DIRECTORY_URL)
    if not first_soup:
        raise RuntimeError("Cannot reach the IIT directory — check your connection.")

    last_link = first_soup.select_one('a[title="Go to last page"]')
    last_page = 0
    if last_link:
        m = re.search(r"page=(\d+)", last_link.get("href", ""))
        if m:
            last_page = int(m.group(1))
    total_pages = last_page + 1
    print(f"📄  {total_pages} listing pages found.")

    # Step 1 — scrape all listing pages concurrently
    all_people: list[dict] = []
    seen_urls:  set[str]   = set()

    for p in parse_listing_page(first_soup):
        if p["Source_url"] not in seen_urls:
            seen_urls.add(p["Source_url"])
            all_people.append(p)

    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = {pool.submit(fetch_listing_page, n): n for n in range(1, total_pages)}
        with tqdm(total=total_pages - 1, desc="Listing pages", unit="pg") as bar:
            for fut in as_completed(futures):
                for p in fut.result():
                    if p["Source_url"] not in seen_urls:
                        seen_urls.add(p["Source_url"])
                        all_people.append(p)
                bar.update(1)

    print(f"✅  {len(all_people)} unique people found.")

    # Step 2 — enrich each profile concurrently
    checkpoint = load_checkpoint() if args.resume else {}
    if args.resume:
        print(f"⏩  Resuming — {len(checkpoint)} profiles already cached.")

    todo = []
    for person in all_people:
        if person["Source_url"] in checkpoint:
            person.update(checkpoint[person["Source_url"]])
        else:
            todo.append(person)

    print(f"🌐  Fetching {len(todo)} profiles ({args.workers} workers) …")
    lock = threading.Lock()

    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = {pool.submit(fetch_profile, p): p for p in todo}
        with tqdm(total=len(todo), desc="Profiles", unit="profile") as bar:
            for fut in as_completed(futures):
                enriched = fut.result()
                with lock:
                    checkpoint[enriched["Source_url"]] = dict(enriched)
                bar.update(1)

    save_checkpoint(checkpoint)

    # Step 3 — deduplicate and write CSV
    seen = set()
    unique = []
    for p in all_people:
        key = p.get("Source_url") or p.get("Name", "")
        if key not in seen:
            seen.add(key)
            unique.append(p)

    with open(args.output, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(unique)

    elapsed = time.time() - t0
    mins, secs = divmod(int(elapsed), 60)
    print(f"\n🎉  Done in {mins}m {secs}s  —  {len(unique)} records  →  {args.output}")
    print(f"    Checkpoint saved to {args.checkpoint}  (set resume=True to continue after interruption)")


if __name__ == "__main__":
    main()

🔍  Fetching page 1 to discover total pages …
📄  142 listing pages found.


Listing pages: 100%|██████████| 141/141 [00:17<00:00,  7.87pg/s]


✅  1345 unique people found.
🌐  Fetching 1345 profiles (10 workers) …


Profiles: 100%|██████████| 1345/1345 [02:58<00:00,  7.55profile/s]


🎉  Done in 3m 16s  —  1345 records  →  iit_people_directory.csv
    Checkpoint saved to iit_checkpoint.json  (set resume=True to continue after interruption)
